In [ ]:
import os
import re

BASE_PATH = "C:/Users/panaa/Desktop/dipseer"

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR = os.path.join(BASE_PATH, "val")
TEST_DIR = os.path.join(BASE_PATH, "test")

DATASET_SPLITS = [TRAIN_DIR, VAL_DIR, TEST_DIR]

# Target frame rate after downsampling
TARGET_FPS = 2
FRAME_INTERVAL = 1.0 / TARGET_FPS

# Expected filename format: HH_MM_SS_microseconds.json
FILENAME_REGEX = re.compile(r"^(\d{2})_(\d{2})_(\d{2})_(\d{6})\.json$", re.IGNORECASE)

In [ ]:
from datetime import datetime, timedelta


def extract_timestamp(filename):
    """
    Extracts a timestamp from a filename in HH_MM_SS_microseconds.json format.
    """

    match = FILENAME_REGEX.match(filename)

    if not match:
        return None

    hours, minutes, seconds, microseconds = match.groups()

    try:
        return datetime(
            year=2000,
            month=1,
            day=1,
            hour=int(hours),
            minute=int(minutes),
            second=int(seconds),
            microsecond=int(microseconds)
        )
    except ValueError:
        return None

In [ ]:
def find_metadata_directories(root_path):
    """
    Finds all directories named 'metadata' recursively within the given root path.
    """

    metadata_dirs = []

    for current_root, _, _ in os.walk(root_path):
        if os.path.basename(current_root).lower() == "metadata":
            metadata_dirs.append(current_root)

    return metadata_dirs

In [ ]:
def load_metadata_files(metadata_dir):
    """
    Loads valid metadata JSON files from a directory and returns them sorted by timestamp.
    """

    files = []

    try:
        entries = os.listdir(metadata_dir)
    except OSError:
        return []

    for filename in entries:
        path = os.path.join(metadata_dir, filename)

        if not os.path.isfile(path):
            continue

        if not filename.lower().endswith(".json"):
            continue

        timestamp = extract_timestamp(filename)

        if timestamp is None:
            continue

        files.append((filename, path, timestamp))

    files.sort(key=lambda x: x[2])

    return files

In [ ]:
from bisect import bisect_left


def select_frame_indices(files, interval):
    """
    Selects frame indices to achieve uniform temporal sampling based on a target interval.
    """

    if not files:
        return set()

    timestamps = [item[2] for item in files]
    num_files = len(files)

    start_time = timestamps[0]
    end_time = timestamps[-1]

    selected_indices = set()
    step = timedelta(seconds=interval)

    target_time = start_time

    while target_time <= end_time:
        pos = bisect_left(timestamps, target_time)

        candidate_indices = []

        if pos < num_files:
            candidate_indices.append(pos)

        if pos - 1 >= 0:
            candidate_indices.append(pos - 1)

        candidate_indices = list(dict.fromkeys(candidate_indices))

        if candidate_indices:
            best_index = min(
                candidate_indices,
                key=lambda idx: abs((timestamps[idx] - target_time).total_seconds())
            )

            if best_index in selected_indices:
                window = 1
                replacement_index = None

                while True:
                    expanded_candidates = []

                    left = pos - window
                    right = pos + window

                    if 0 <= left < num_files:
                        expanded_candidates.append(left)

                    if 0 <= right < num_files:
                        expanded_candidates.append(right)

                    expanded_candidates = [
                        idx for idx in dict.fromkeys(expanded_candidates)
                        if idx not in selected_indices
                    ]

                    if expanded_candidates:
                        replacement_index = min(
                            expanded_candidates,
                            key=lambda idx: abs((timestamps[idx] - target_time).total_seconds())
                        )
                        break

                    if left < 0 and right >= num_files:
                        break

                    window += 1

                if replacement_index is not None:
                    best_index = replacement_index
                else:
                    target_time += step
                    continue

            selected_indices.add(best_index)

        target_time += step

    return selected_indices

In [ ]:
def reduce_metadata_fps(metadata_dir):
    files = load_metadata_files(metadata_dir)

    if not files:
        return

    selected_indices = select_frame_indices(files, FRAME_INTERVAL)
    selected_filenames = {files[i][0] for i in selected_indices}

    for name, path, _ in files:
        if name not in selected_filenames:
            try:
                os.remove(path)
            except OSError:
                pass

In [ ]:
def process_dataset(root_path):
    metadata_dirs = find_metadata_directories(root_path)

    for metadata_dir in metadata_dirs:
        reduce_metadata_fps(metadata_dir)

In [ ]:
def run():
    for dataset_split in DATASET_SPLITS:
        process_dataset(dataset_split)

In [ ]:
if __name__ == "__main__":
    run()